# MicroDuck recurrent PPO with backend-neutral MuJoCo

This notebook trains a small forward-locomotion task on a laptop CPU with the recurrent PPO pipeline from `ppo_mujoco.py`: a GRU actor-critic initialized at a validated closed-form gait, a collector that writes whole episodes into a replay buffer, and whole-episode PPO minibatches. The task is `torchrl.envs.MicroDuckEnv`; changing `BACKEND` to `"mjx"` or `"mujoco-torch"` changes the physics implementation only.

From the TorchRL checkout, start Jupyter with:

```bash
uv run --extra rendering --extra mujoco_wasm --extra notebook --with mujoco --with matplotlib \
  jupyter lab examples/microduck/microduck_ppo.ipynb
```

The `mujoco_wasm` extra supplies a Node.js runtime for the final MuJoCo WASM viewer cell.

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchrl.envs import ExplorationType, MicroDuckEnv, set_exploration_type
from torchrl.envs.utils import check_env_specs
from torchrl.render import (
    display_mujoco_wasm_viewer,
    play_mujoco_wasm_trajectory,
    write_mujoco_wasm_viewer,
)

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "torchrl").is_dir() and (path / "examples").is_dir()
)
sys.path.insert(0, str(repo_root))

from examples.microduck.ppo_mujoco import (
    evaluate_policy,
    make_env,
    make_models,
    train_ppo,
)

## Locate the shared MJCF

The XML and meshes stay in `microduck_rl`; nothing is copied or translated. Set `MICRODUCK_RL_ROOT` to an existing checkout. Otherwise the cell downloads a pinned source archive next to the TorchRL checkout, without needing Git.

In [ ]:
MICRODUCK_RL_COMMIT = "d424a0c899f6b33cbd3daeb279913134349c0b63"

microduck_root = Path(
    os.environ.get("MICRODUCK_RL_ROOT", repo_root.parent / "microduck_rl")
).expanduser()
if not microduck_root.exists():
    microduck_root.parent.mkdir(parents=True, exist_ok=True)
    archive_url = (
        "https://github.com/pollen-robotics/microduck_rl/archive/"
        f"{MICRODUCK_RL_COMMIT}.zip"
    )
    with tempfile.TemporaryDirectory(prefix="microduck_rl_download_") as tmp_dir:
        archive_path = Path(tmp_dir) / "microduck_rl.zip"
        urllib.request.urlretrieve(archive_url, archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(tmp_dir)
        shutil.move(Path(tmp_dir) / f"microduck_rl-{MICRODUCK_RL_COMMIT}", microduck_root)
scene_path = MicroDuckEnv.resolve_scene(microduck_root)
scene_path

## The environment

The action is a 14-dimensional normalized joint-position offset around the MJCF `STAND` keyframe. The 53-dimensional observation is projected gravity (3), body angular velocity (3), measured and commanded body-frame forward velocity (2), joint-position error (14), joint velocity (14), the gait clock (3), and the previous action (14). `make_env` adds the `InitTracker` and recurrent-state primer the GRU policy needs; native MuJoCo uses `SerialEnv` so the batch stays in the notebook kernel.

In [ ]:
BACKEND = "mujoco"  # "mujoco", "mjx", or "mujoco-torch"
NUM_ENVS = 4
HIDDEN_SIZE = 64
COMMANDS = (0.03,)

env = make_env(
    scene_path,
    backend=BACKEND,
    commanded_x_velocity=COMMANDS,
    num_envs=NUM_ENVS,
    hidden_size=HIDDEN_SIZE,
    seed=0,
)
check_env_specs(env)
reset_td = env.reset()
reset_td["observation"].shape, reset_td["commanded_x_velocity"].flatten(), env.action_spec.shape

## Native render before training

Rendering uses the official MuJoCo renderer on CPU with an external camera. Training keeps pixels out of the hot loop.

In [ ]:
render_env = MicroDuckEnv(
    scene_path,
    num_envs=1,
    reset_noise_scale=0.0,
    render_width=640,
    render_height=480,
    camera_id=-1,
)
render_env.reset()
frame = render_env.render()[0].cpu()
plt.figure(figsize=(8, 6))
plt.imshow(frame)
plt.axis("off")
plt.show()

## A short PPO run

This is a pipeline check, not a locomotion benchmark: the command-line script defaults to 10 million transitions with 16,384 transitions per update. The first policy is the closed-form gait, so evaluation at transition 0 already walks; PPO learns a bounded residual around it.

In [ ]:
torch.manual_seed(0)
actor, critic = make_models(env, hidden_size=HIDDEN_SIZE)
evaluation_env = make_env(
    scene_path, backend=BACKEND, num_envs=1, hidden_size=HIDDEN_SIZE, seed=1
)
history = train_ppo(
    env,
    actor,
    critic,
    total_transitions=4_000,
    transitions_per_update=2_000,
    epochs=3,
    minibatch_trajectories=4,
    evaluation_env=evaluation_env,
    evaluation_interval=1,
    evaluation_commands=COMMANDS,
    evaluation_seeds=(0, 1),
    evaluation_steps=200,
)
history[-1]

In [ ]:
plt.plot(
    [row["progress/transitions"] for row in history],
    [row["episode/return_mean"] for row in history],
)
plt.xlabel("transitions")
plt.ylabel("mean episode return")
plt.show()

evaluate_policy(
    evaluation_env, actor, commanded_x_velocities=COMMANDS, seeds=(0, 1), steps=200
)

## Interactive MuJoCo WASM viewer

`torchrl.render` copies the original XML, its includes and meshes into a local browser viewer, so the duck looks like the real robot even though physics ran with box collision proxies. The iframe exposes the 14 joint sliders and Reset, Step and Run controls, and plays the `qpos` trajectory collected from the policy. Moving a slider interrupts playback.

In [ ]:
qpos_trajectory = []
with torch.no_grad(), set_exploration_type(ExplorationType.DETERMINISTIC):
    td = evaluation_env.reset()
    qpos_trajectory.append(evaluation_env.base_env.get_state()["qpos"][0].tolist())
    for _ in range(300):
        actor(td)
        td = evaluation_env.step(td)["next"]
        qpos_trajectory.append(evaluation_env.base_env.get_state()["qpos"][0].tolist())
        if bool(td["done"].any()):
            break
len(qpos_trajectory), len(qpos_trajectory[0])

In [ ]:
viewer_dir = Path(tempfile.mkdtemp(prefix="torchrl_microduck_wasm_"))
write_mujoco_wasm_viewer(viewer_dir, scene_path)
viewer_process = display_mujoco_wasm_viewer(viewer_dir, height=720)
play_mujoco_wasm_trajectory(
    qpos_trajectory,
    fps=30,
    loop=True,
    viewer_dir=viewer_dir,
    viewer_origin=viewer_process.torchrl_mujoco_wasm_origin,
)

In [ ]:
viewer_process.terminate()
evaluation_env.close()
render_env.close()
env.close()